In [0]:
%sql
INSERT INTO dev.mohit_gangwani.metrics_population_report_onn
WITH tvs AS (
  SELECT tvid, token
  FROM prod.detection.tv
  WHERE oem = 'ONN'
)
, viewing_sessions as (
	SELECT tvs.token, vc.fk_content_id, DATE(vc.session_start) AS session_start
	FROM prod.detection_onn.viewing_content_firehose vc
  JOIN tvs ON tvid = vc.fk_tvid
	WHERE fk_tvid IN (SELECT DISTINCT tvid FROM prod.detection.tv_zoo_latest_daily WHERE zoo = 'control-zoo-dtsprod.tvinteractive.tv')
	  AND session_start >= CURRENT_DATE - INTERVAL '31 DAYS'
    AND session_start < CURRENT_DATE
  GROUP BY ALL
),
ispot_commercials as (
	SELECT tvs.token, DATE(session_start) AS session_start
	FROM prod.detection_onn.viewing_commercials_firehose AS vc
  JOIN tvs ON tvid = vc.fk_tvid
	JOIN prod.detection.tv_zoo_latest_daily AS zoo
	  ON vc.fk_tvid = zoo.tvid
	LEFT JOIN prod.detection.commercial_id_external_firehose AS cie
	  ON vc.fk_commercial_id = cie.fk_commercial_id
	WHERE cie.fk_client_id = 8
	  AND vc.session_start >= CURRENT_DATE - INTERVAL '31 DAYS'
    AND vc.session_start < CURRENT_DATE
	  AND zoo.zoo_id = 17
  GROUP BY ALL
),
one_year_opted_in as (
  SELECT '1 year active (opted in)' as category
  , CURRENT_DATE as create_timestamp
  , COUNT(DISTINCT tv.token) as one_year_active
  FROM prod.detection.tv_activity ta
  JOIN tvs tv
    ON tv.tvid = ta.fk_tvid
  JOIN prod.detection.tv_terms_of_service tos
    ON tos.fk_tvid = ta.fk_tvid
   AND tos.create_timestamp <= ta.session_start
   AND tos.next_create_timestamp > ta.session_start
  JOIN prod.detection.tv_settings tvst
    ON tvst.fk_tvid = ta.fk_tvid
   AND tvst.create_timestamp <= ta.session_start
   AND tvst.next_create_timestamp > ta.session_start
  JOIN prod.detection.settings st
    ON st.settings_id = tvst.fk_settings_id
  WHERE ta.session_end >= CURRENT_DATE - INTERVAL '366 DAYS'
    AND ta.session_start < CURRENT_DATE
    AND tos.tos_version >= 514
    AND tos.next_create_timestamp >=  TIMESTAMPADD(DAY, -7, DATE_TRUNC('WEEK', CURRENT_DATE - INTERVAL '366 DAYS'))
    AND st.disabled = 0
    AND st.points_allowed = 1
    AND st.country_name = 'USA'
    AND tvst.next_create_timestamp >=  TIMESTAMPADD(DAY, -7, DATE_TRUNC('WEEK', CURRENT_DATE - INTERVAL '366 DAYS'))
),
thirty_day_reporting as (
  SELECT CURRENT_DATE as create_timestamp,
    COUNT(DISTINCT token) as thirty_day_reporting
    FROM viewing_sessions
),
thirty_day_detecting as (
  SELECT CURRENT_DATE as create_timestamp
  , COUNT(DISTINCT token) as thirty_day_detecting
  FROM viewing_sessions
  WHERE fk_content_id != 3468026 --3468026 is null detection
),
one_day_detecting as (
  SELECT CURRENT_DATE as create_timestamp
  , COUNT(DISTINCT token) as one_day_detecting
  FROM viewing_sessions
  WHERE fk_content_id != 3468026
    AND session_start >= CURRENT_DATE - INTERVAL '1 DAY'
    AND session_start < CURRENT_DATE
  GROUP BY 1
),
thirty_day_ispot as (
  SELECT CURRENT_DATE as create_timestamp
  , COUNT(DISTINCT token) as thirty_day_ispot_detecting
  FROM ispot_commercials
  GROUP BY 1
),
one_day_ispot as (
  SELECT CURRENT_DATE as create_timestamp
  , COUNT(DISTINCT token) as one_day_ispot_detecting
  FROM ispot_commercials
  WHERE session_start >= CURRENT_DATE - INTERVAL '1 DAYS'
    AND session_start < CURRENT_DATE
  GROUP BY 1
)
SELECT a.create_timestamp
, a.one_year_active
, b.thirty_day_reporting
, c.thirty_day_detecting
, d.one_day_detecting
, e.thirty_day_ispot_detecting
, f.one_day_ispot_detecting
FROM one_year_opted_in a
JOIN thirty_day_reporting b
  ON a.create_timestamp = b.create_timestamp
JOIN thirty_day_detecting c
  ON a.create_timestamp = c.create_timestamp
JOIN one_day_detecting d
  ON a.create_timestamp = d.create_timestamp
JOIN thirty_day_ispot e
  ON a.create_timestamp = e.create_timestamp
JOIN one_day_ispot f
  ON a.create_timestamp = f.create_timestamp